# Ingest data to Bronze zone

- Spark reads database (postgres) and write (partition by date) to parquet file in Bronze zone

In [3]:
from pyspark.sql import SparkSession
import os
import pyspark.sql.functions as F
from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

- Config Spark để thực hiện một số việc sau:
    - Đọc/Ghi trên S3: Cần phải truyền vào credential của AWS tương ứng với account. Việc này có thể gặp một số lỗi do conflict policy trên AWS. Ví dụ: 1 user vừa thuộc group admin (full quyền), vừa thuộc group dev (không có quyền tạo bucket trên S3). Lúc này AWS sẽ ưu tiên các quyền bị chặn. Cần remove user ra khỏi group có quyền bị chặn
    - Đọc/Ghi Postgres: Truyền vào file jar tương ứng
    - Vì tôi đang ingest data vào Bronze zone (which is a data lake architecture), tôi phải đảm bảo các hoạt động của tôi được ghi lại bởi Hive. Cụ thể, tôi cần config endpoint của Hive, đảm bảo Spark tải đúng config version của Hive từ maven, và phải cung cấp một location để lưu thông tin schema (đoạn này thì tôi không rõ những nếu không cung cấp location thì khi tạo schema từ spark, sẽ bị lỗi)

In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to bronze zone')
    # read/write S3
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.4.2,software.amazon.awssdk:bundle:2.29.38")
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # read postgres
    .config('spark.jars', '../../spark/jar/postgresql-42.7.11.jar')
    # Hive metastore on localhost
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9083")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.hive.metastore.version", "3.1.3")
    .config("spark.sql.hive.metastore.jars", "maven")
    # create schema using spark (instead of using trino)
    .config("spark.sql.warehouse.dir", "s3a://flezi-data/warehouse/")
    .enableHiveSupport()
    .getOrCreate()
)
spark

:: loading settings :: url = jar:file:/home/longnb32/FPT/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/longnb32/.ivy2.5.2/cache
The jars for the packages stored in: /home/longnb32/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db8847d8-32cb-4a64-93b0-c98238c1c13d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 301ms :: artifacts dl 9ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.1.4.Final from central in [def

In [3]:
# create schema in trino
spark.sql('create database if not exists taxi_data')

https://maven-central.storage-download.googleapis.com/maven2/ added as a remote repository with the name: repo-1
https://maven-central.storage-download.googleapis.com/maven2/ added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/home/longnb32/FPT/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/longnb32/.ivy2.5.2/cache
The jars for the packages stored in: /home/longnb32/.ivy2.5.2/jars
org.apache.logging.log4j#log4j-api added as a dependency
org.apache.derby#derby added as a dependency
org.apache.hive#hive-metastore added as a dependency
org.apache.hive#hive-exec added as a dependency
org.apache.hive#hive-common added as a dependency
org.apache.hive#hive-serde added as a dependency
org.apache.hadoop#hadoop-client-api added as a dependency
org.apache.hadoop#hadoop-client-runtime added as a dependency
:: resolving dependencies :: org.apache.spark#spark-

DataFrame[]

In [3]:
df = (
    spark.read.format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", f"jdbc:postgresql://localhost:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_DB_NAME')}")
    .option("dbtable", "public.nyc_taxi")
    .option("user", os.getenv("COM_DB_USER"))
    .option("password", os.getenv("COM_DB_PASSWORD"))
    .load()
)
df.count()

140120

In [5]:
month_partition_df = (
    df.withColumn('year', F.year('pickup_datetime'))
    .withColumn('month', F.month('pickup_datetime'))
)
month_partition_df.show(10)

26/05/29 09:47:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+----+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|year|month|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------

In [6]:
S3_BRONZE_PATH = "s3a://flezi-data/bronze/"
SCHEMA = "taxi_data"
TABLE  = "nyc_taxi"

(
    month_partition_df
    .write
    .partitionBy("year", "month")
    .mode("append")
    .option("path", S3_BRONZE_PATH)
    .saveAsTable(f"{SCHEMA}.{TABLE}")   # đăng ký luôn với Hive metastore
)

print(f"Done. Table registered as '{SCHEMA}.{TABLE}' → {S3_BRONZE_PATH}")

26/05/29 09:48:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://flezi-data/bronze.
java.io.FileNotFoundException: No such file or directory: s3a://flezi-data/bronze
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AFileSystem.java:2917)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AFileS

Done. Table registered as 'taxi_data.nyc_taxi' → s3a://flezi-data/bronze/


- Sau khi ghi xuống DB xong thì phải lưu lại latest pickup date, để lần sau sẽ dựa vào đó để xử lý tiếp (chỉ lấy những record có pickup date sau latest pickup date)

In [4]:
latest_pickup_date = df.select(F.max('pickup_datetime')).collect()
latest_pickup_date

[Row(max(pickup_datetime)=datetime.datetime(2009, 1, 31, 23, 59, 55))]

In [5]:
str(latest_pickup_date[0][0])

'2009-01-31 23:59:55'

---

In [28]:
latest_pickup_date = df.select(F.max('pickup_datetime')).collect()
latest_pickup_date

[Row(max(pickup_datetime)=None)]

In [29]:
df = (
    spark.read.format("jdbc")
    .option("driver", "org.postgresql.Driver")
    .option("url", f"jdbc:postgresql://localhost:{os.getenv('COM_DB_PORT')}/{os.getenv('COM_DB_NAME')}")
    .option("dbtable", "public.nyc_taxi")
    .option("user", os.getenv("COM_DB_USER"))
    .option("password", os.getenv("COM_DB_PASSWORD"))
    .load()
)

# df = df.filter(
#     df['pickup_datetime'] > str(latest_pickup_date[0][0])
# )

df = df.filter(
    df['pickup_datetime'] > '2009-08-28'
)

df.count()

157066

In [30]:
month_partition_df = (
    df.withColumn('year', F.year('pickup_datetime'))
    .withColumn('month', F.month('pickup_datetime'))
)

(
    month_partition_df
    .write
    .partitionBy("year", "month")
    .mode("append")
    .option("path", S3_BRONZE_PATH)
    .saveAsTable(f"{SCHEMA}.{TABLE}")   # đăng ký luôn với Hive metastore
)

print(f"Done. Table registered as '{SCHEMA}.{TABLE}' → {S3_BRONZE_PATH}")

Done. Table registered as 'taxi_data.nyc_taxi' → s3a://flezi-data/bronze/


---

In [17]:
df.filter(df['pickup_datetime'] == latest_pickup_date[0][0]).collect()

[Row(vendor_id='VTS', pickup_datetime=datetime.datetime(2009, 7, 27, 19, 16), dropoff_datetime=datetime.datetime(2009, 7, 27, 19, 25), passenger_count=1.0, trip_distance=1.33, rate_code_id=None, store_and_fwd_flag=None, pickup_longitude=-73.967523, pickup_latitude=40.766052, dropoff_longitude=-73.950777, dropoff_latitude=40.77104, pu_location_id=None, do_location_id=None, payment_type='CASH', fare_amount=6.9, extra=None, mta_tax=None, surcharge=1.0, improvement_surcharge=None, tip_amount=0.0, tolls_amount=0.0, total_amount=7.9, congestion_surcharge=None, airport_fee=None, cbd_congestion_fee=None),
 Row(vendor_id='VTS', pickup_datetime=datetime.datetime(2009, 7, 27, 19, 16), dropoff_datetime=datetime.datetime(2009, 7, 27, 19, 22), passenger_count=2.0, trip_distance=0.7399999999999999, rate_code_id=None, store_and_fwd_flag=None, pickup_longitude=-73.974608, pickup_latitude=40.757858, dropoff_longitude=-73.982827, dropoff_latitude=40.761538, pu_location_id=None, do_location_id=None, payme

In [9]:
str(latest_pickup_date[0][0])

'2009-07-27 19:16:00'

In [12]:
last_month_df = df.filter(
    (df['pickup_datetime'] < str(latest_pickup_date[0][0])) &
    (df['pickup_datetime'] > '2009-06-27 19:16:00')
)
print(last_month_df.count())
last_month_df.show()

133293
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+--

In [19]:
last_month_df.select(F.min('pickup_datetime')).collect()[0][0]

datetime.datetime(2009, 6, 27, 19, 16, 41)